# LSTM v02 — node-only với shared selected features

Mỗi service được xử lý độc lập bởi node MLP và LSTM dùng chung trọng số. Mô hình chỉ nhận 10 node features của chính service đó và service embedding; không nối dữ liệu của 11 service, không dùng graph hoặc edge features. Độ sâu/rộng MLP khớp với GAT encoder để tạo đối chứng capacity-oriented cho GAT-LSTM.

Quy trình: train trọng số trên train, chọn checkpoint và hyperparameters bằng validation, sau đó mới đánh giá 5 seed trên frozen test. Artifact joint LSTM v01 được giữ nguyên để bảo toàn kết quả lịch sử.

In [1]:
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa

ctx = fa.load_context(ROOT)
selection = fa.load_selection_contract(
    ROOT / 'configs' / 'gatgru' / 'feature_contract_v01.json',
    require_edge=False,
)
base_cfg = fa.load_baseline_config(ctx)
base_cfg['use_edge_features'] = False
base_cfg['emb_dim'] = 4

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = (ROOT / 'artifacts' / 'lstm' /
       'v02_node_only_selected_features')

print('shared node features:', selection['node_features'])
print('model kind: lstm_node_only')
print('edge features: []')
print('device:', DEVICE)
print('output:', OUT)

shared node features: ['rps_in', 'rps_in_delta', 'rps_per_replica', 'latency_p50_ms', 'latency_p95_ms', 'cpu_cores', 'mem_mib', 'net_rx_kBps', 'net_tx_kBps', 'replicas']
model kind: lstm_node_only
edge features: []
device: cpu
output: F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\lstm-load-testing\experiments\lstm\v02_node_only_selected_features


## 1. Tune trên validation với kiến trúc node-only được khóa

Node MLP, LSTM hidden size và embedding được giữ cố định theo graph-temporal baseline. Optuna chỉ chọn window, dropout, learning rate, weight decay và batch size. Test chưa được truy cập ở bước này.

In [2]:
best_cfg, study = fa.run_optuna_fixed_architecture(
    ctx,
    'lstm_node_only',
    selection['node_features'],
    [],
    base_cfg,
    OUT / 'tuning',
    n_trials=24,
    seed=42,
    device=DEVICE,
    score_weights=fa.CPU_DIAGNOSTIC_WEIGHTS,
    parent_version='v01_shared_selected_features_joint',
)
assert best_cfg['use_edge_features'] is False
print('best validation score:', study.best_value)
best_cfg

[I 2026-10-25 23:35:18,840] Using an existing study with `study_name='lstm_node_only_b55e5fc117df47d5_fixed_arch'` instead of creating a new one.
[I 2026-10-25 23:47:32,885] Trial 12 finished with value: 0.10803011861954623 and parameters: {'window': 18, 'dropout': 0.11973169683940732, 'lr': 0.0006427207987637223, 'weight_decay': 2.9375384576328313e-06, 'batch_size': 64}. Best is trial 8 with value: 0.10781314312178239.
[I 2026-10-25 23:54:33,285] Trial 13 finished with value: 0.10915810394939598 and parameters: {'window': 18, 'dropout': 0.15466041019831017, 'lr': 0.0019848040173260323, 'weight_decay': 2.0887804157145047e-05, 'batch_size': 64}. Best is trial 8 with value: 0.10781314312178239.
[I 2026-10-26 00:05:40,390] Trial 14 finished with value: 0.10796728317547538 and parameters: {'window': 24, 'dropout': 0.1248911480136476, 'lr': 0.0012319351011309033, 'weight_decay': 2.712741657054576e-06, 'batch_size': 64}. Best is trial 8 with value: 0.10781314312178239.
[I 2026-10-26 00:14:40

best validation score: 0.1076088841884564


{'window': 24,
 'dropout': 0.13961092233707767,
 'lr': 0.0011888166798423134,
 'batch_size': 64,
 'gat_hidden': 32,
 'heads': 4,
 'gat_layers': 2,
 'gru_hidden': 128,
 'edge_direction': 'both',
 'use_edge_features': False,
 'emb_dim': 4,
 'weight_decay': 0.00023718306892852383}

## 2. Huấn luyện 5 seed và đánh giá frozen test

Mỗi seed vẫn huấn luyện trên train và dùng validation để chọn checkpoint. `run_final` không refit trên train + validation.

In [3]:
prediction_path, training_meta = fa.run_final(
    ctx,
    'lstm_node_only',
    selection['node_features'],
    [],
    best_cfg,
    OUT,
    seeds=(0, 1, 2, 3, 4),
    device=DEVICE,
    model_name='lstm_node_only',
    parent_version='v01_shared_selected_features_joint',
)
assert training_meta['model_kind'] == 'lstm_node_only'
print(prediction_path)
training_meta

F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\lstm-load-testing\experiments\lstm\v02_node_only_selected_features\lstm_node_only_predictions_test.npz


{'model': 'lstm_node_only',
 'model_kind': 'lstm_node_only',
 'config': {'window': 24,
  'dropout': 0.13961092233707767,
  'lr': 0.0011888166798423134,
  'batch_size': 64,
  'gat_hidden': 32,
  'heads': 4,
  'gat_layers': 2,
  'gru_hidden': 128,
  'edge_direction': 'both',
  'use_edge_features': False,
  'emb_dim': 4,
  'weight_decay': 0.00023718306892852383},
 'seeds': [0, 1, 2, 3, 4],
 'val_mae_scaled': [0.11645521968603134,
  0.1150798574090004,
  0.11478013545274734,
  0.11636489629745483,
  0.11667092889547348],
 'node_features': ['rps_in',
  'rps_in_delta',
  'rps_per_replica',
  'latency_p50_ms',
  'latency_p95_ms',
  'cpu_cores',
  'mem_mib',
  'net_rx_kBps',
  'net_tx_kBps',
  'replicas'],
 'edge_features': [],
 'targets': ['rps_in', 'cpu_cores'],
 'services': ['frontend',
  'adservice',
  'cartservice',
  'checkoutservice',
  'currencyservice',
  'emailservice',
  'paymentservice',
  'productcatalogservice',
  'recommendationservice',
  'redis-cart',
  'shippingservice'],
 'n